# Clean MercuryDailyEvents

Cleans a raw `MercuryDailyEvents_yyyy-mm-dd.txt` export.

## Imports

In [ ]:
import csv
import io
import re
from pathlib import Path

import pandas as pd

## Schema constants

- `LS_COLS` — the final column set and order for the cleaned output.
- `LS_STRING_COLS` — the free-text columns that get whitespace-trimmed and have internal double-whitespace collapsed to a single space.
- `LS_INT_COLS` — the numeric columns that get cast to integer type.
- `RENAME_MAP` — raw → cleaned column renames (see the note at the top of this notebook).
- `SORT_COLS` — the columns (and order) used for the final ascending sort (the notes call this `SORT_ORDER`).
- `FILENAME_RE` — extracts the year/month from the input filename, used both to build `month_tag` and to auto-detect the input file.


In [ ]:
LS_COLS = [
    "Date", "CompanyCode", "CompanyName", "Country", "Operation", "EventType", "DeviceCategory",
    "SearchTerm", "ContentType", "ContentTitle", "DownloadLanguage", "Theme", "Route",
    "UniqueUsers", "DateRange",
]
LS_STRING_COLS = [
    "CompanyCode", "CompanyName", "Country", "Operation", "EventType", "DeviceCategory",
    "SearchTerm", "ContentType", "ContentTitle", "DownloadLanguage", "Theme", "Route",
    "DateRange",
]
LS_INT_COLS = ["UniqueUsers"]

RENAME_MAP = {
    "ContentTitleEN": "ContentTitle",
    "downloadLanguage": "DownloadLanguage",
    "uniqueUsers": "UniqueUsers",
}

SORT_COLS = [
    "Date", "CompanyCode", "CompanyName", "Country", "Operation", "EventType",
    "DeviceCategory", "SearchTerm", "ContentType", "ContentTitle",
]

FILENAME_RE = re.compile(r"^MercuryDailyEvents_(\d{4})-(\d{2})-\d{2}\.txt$")

## Locate the input file

`find_default_input` looks for a single `MercuryDailyEvents_yyyy-mm-dd.txt` file in a given directory and returns it automatically. If none or several are found, it raises rather than silently guessing which one to use.


In [ ]:
def find_default_input(directory: Path) -> Path:
    matches = sorted(p for p in directory.glob("MercuryDailyEvents_*.txt") if FILENAME_RE.match(p.name))
    if not matches:
        raise FileNotFoundError(f"No MercuryDailyEvents_yyyy-mm-dd.txt file found in {directory}")
    if len(matches) > 1:
        raise ValueError(
            f"Multiple candidate input files found in {directory}: "
            f"{[m.name for m in matches]}. Pass one explicitly."
        )
    return matches[0]


## Derive `month_tag` from the filename

The output name has the format of `MercuryDailyEvents_<month_tag>_cleaned.csv`, where `month_tag` is `yyyymm` for the month *before* the input filename's `yyyy-mm-dd` date suffix (the export date's month minus one), per the notes' worked example.


In [ ]:
def month_tag_from_filename(path: Path) -> str:
    match = FILENAME_RE.match(path.name)
    if not match:
        raise ValueError(f"Filename '{path.name}' does not match expected pattern MercuryDailyEvents_yyyy-mm-dd.txt")
    year, month = (int(g) for g in match.groups())
    # month_tag refers to the prior month's data, not the export date's month.
    year, month = (year - 1, 12) if month == 1 else (year, month - 1)
    return f"{year}{month:02d}"


## Read the raw file, protecting genuine backslashes

This dataset needs `engine="python", escapechar="\\"` to parse genuine `\"..\"` escapes around quoted phrases inside `SearchTerm`/`ContentTitle` (e.g. `\"daily wellness check-in\"`). Left unguarded, `escapechar` strips *every* backslash it precedes, not just ones before a quote — so a literal backslash in `CompanyCode`/`CompanyName` (e.g. a client named `TBWA\RAAD`) would be silently corrupted to `TBWARAAD`. `read_semicolon_csv_protecting_backslashes` pre-processes the raw text to double any backslash *not* immediately followed by `"`, so `escapechar` only ever consumes genuine `\"` sequences and every other backslash survives intact.

In [ ]:
def read_semicolon_csv_protecting_backslashes(path: Path) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        raw_text = f.read()

    # Protect literal backslashes that aren't a genuine CSV \" escape by doubling them,
    # so escapechar only ever consumes actual \" sequences below.
    protected_text = re.sub(r'\\(?!")', r"\\\\", raw_text)

    return pd.read_csv(
        io.StringIO(protected_text), sep=";", engine="python", escapechar="\\",
        dtype=str, keep_default_na=False, encoding="utf-8",
    )

## Cleaning logic

The core transformation, in the order implemented:

1. Rename `ContentTitleEN`→`ContentTitle`, `downloadLanguage`→`DownloadLanguage`, `uniqueUsers`→`UniqueUsers`.
2. Drop rows that are blank across every `LS_COLS` field present in the raw data.
3. Drop rows where `Date` is blank.
4. Reformat `Date` to `%Y-%m-%d %H:%M:%S.%f` truncated to 3 decimals (milliseconds).
5. Reorder/drop columns to match `LS_COLS`.
6. Trim surrounding whitespace, then collapse any internal run of whitespace to a single space, on the `LS_STRING_COLS` fields.
7. Cast `UniqueUsers` to integer type.
8. Sort ascending by `Date`, `CompanyCode`, `CompanyName`, `Country`, `Operation`, `EventType`, `DeviceCategory`, `SearchTerm`, `ContentType`, `ContentTitle`.


In [ ]:
def clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns=RENAME_MAP)

    # Same reasoning as the other notebooks: judge "completely blank" against the
    # LS_COLS fields present in the raw data, since raw pipeline-metadata columns
    # (FileName, PipelineRunID, ImportDate, CreatedBy, DataSource, ...) are dropped
    # later and would otherwise mask genuinely blank rows.
    present_ls_cols = [c for c in LS_COLS if c in df.columns]
    is_blank = df[present_ls_cols].apply(lambda col: col.str.strip() == "").all(axis=1)
    df = df.loc[~is_blank].copy()

    missing_key = df["Date"].str.strip() == ""
    df = df.loc[~missing_key].copy()

    # %f always zero-pads to 6-digit microseconds; slicing off the last 3 leaves milliseconds.
    df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d").dt.strftime("%Y-%m-%d %H:%M:%S.%f").str[:-3]

    df = df[LS_COLS]

    for col in LS_STRING_COLS:
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)

    for col in LS_INT_COLS:
        df[col] = df[col].astype(int)

    df = df.sort_values(by=SORT_COLS, ascending=True).reset_index(drop=True)

    return df


## Configure the input file

Leave `INPUT_FILE` as `None` to auto-detect the single raw file in this notebook's `input/` folder, or set it to an explicit path to override (equivalent to the script's optional CLI argument).


In [ ]:
NOTEBOOK_DIR = Path.cwd()
INPUT_FILE = None  # e.g. "input/MercuryDailyEvents_2026-08-02.txt"

input_path = Path(INPUT_FILE).resolve() if INPUT_FILE else find_default_input(NOTEBOOK_DIR / "input")
month_tag = month_tag_from_filename(input_path)
input_path, month_tag


## Read the raw text file

Uses `read_semicolon_csv_protecting_backslashes` (see above) rather than a plain `pd.read_csv`, so `escapechar` only ever consumes genuine `\"..\"` sequences.

In [ ]:
df_raw = read_semicolon_csv_protecting_backslashes(input_path)
df_raw.shape

## Apply the cleaning steps

In [ ]:
df_cleaned = clean(df_raw)
df_cleaned.head()


## Save the cleaned dataset

Written as `;`-delimited UTF-8 with minimal quoting, matching the input file's own semicolon delimiter as required by the notes. Saved to this notebook's `output/` folder.


In [ ]:
output_dir = NOTEBOOK_DIR / "output"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"MercuryDailyEvents_{month_tag}_cleaned.csv"
df_cleaned.to_csv(output_path, sep=";", index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

print(f"Cleaned {len(df_cleaned)} rows -> {output_path}")


## Write summary report

Writes a plain-text report answering: which input file was read, the raw and cleaned row counts, how many duplicate rows exist in each of the raw and cleaned dataframes, the derived `month_tag`, how many rows were dropped for being blank / missing their key field, and the output CSV's name, followed (after three blank lines) by `df_cleaned.describe()`. Saved to this notebook's `reports/` folder as `MercuryDailyEvents_<month_tag>_report.txt`.

Duplicate counts use pandas' default `duplicated()` (`keep="first"`), i.e. the number of rows that are repeats of an earlier row — how many rows would go away if the dataframe were deduplicated.


In [ ]:
report_lines = [
    f"Input file: {input_path.name}",
    f"Raw row count: {len(df_raw)}",
    f"Raw duplicate rows: {int(df_raw.duplicated().sum())}",
    "=======================================================================",
    f"Month tag: {month_tag}",
    f"Blank/missing-key rows dropped: {len(df_raw) - len(df_cleaned)}",
    "=======================================================================",
    f"Cleaned row count: {len(df_cleaned)}",
    f"Cleaned duplicate rows: {int(df_cleaned.duplicated().sum())}",
    f"Output file: {output_path.name}",
]
report_text = "\n".join(report_lines) + "\n"
report_text += "\n\n\n" + df_cleaned.describe().to_string() + "\n"

reports_dir = NOTEBOOK_DIR / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
report_path = reports_dir / f"MercuryDailyEvents_{month_tag}_report.txt"
report_path.write_text(report_text, encoding="utf-8")

print(report_text)
print(f"Report written -> {report_path}")
